# NB01 - Introduction à Spark Structured Streaming

## A. Mise en place du Stream

In [0]:
# Passe la variable 'full_path' au notebook NB01/Setup
import os

full_path = os.getcwd() + "/Resources/NB01/"

dbutils.widgets.text("full_path", full_path)
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "spark_training")

In [0]:
%run "./Resources/NB01/Setup"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# definition du schéma pour le Stream
schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("notifications", StringType(), True),
    StructField("order_id", LongType(), True),
    StructField("order_timestamp", StringType(), True)
])

# utilisation du schéma pour le streaming du dataframe
stream_df = spark.readStream \
    .format("json") \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/workspace/default/spark_training/spark_streaming/data") \
    .load()

In [0]:
print(f"isStreaming: {stream_df.isStreaming}")

## B. Lancement de requêtes de streaming de base

In [0]:
# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming/checkpoint", recurse=True)

display(stream_df, streamName="mon_stream", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming/checkpoint")

### C. Transformations basiques sur le Stream
- select
- filter
- withColumn
- ...

In [0]:
transformed_stream = stream_df \
    .withColumn("notification_status", col("notifications").isNotNull()) \
    .withColumn("order_details", concat(lit("Order #"), col("order_id").cast("string")))

# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming/checkpoint", recurse=True)

display(transformed_stream, streamName="mon_stream", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming/checkpoint")

In [0]:
# filtre sur les lignes avec les notifications activées
notifications_df = stream_df \
    .filter(col("notifications") == "Y")

# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_notif", recurse=True)

display(notifications_df, streamName="stream_notif", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_notif")

### Attention

tout les triggers ne sont pas disponible sur les clusters serverless de databricks

| Trigger | Description | Compatibilité |
|---------|-------------|---------------|
| `trigger(availableNow=True)` | Traite toutes les données disponibles puis s'arrête | ✅ Serverless/Shared |
| `trigger(once=True)` | Traite un micro-batch puis s'arrête | ✅ Serverless/Shared |
| `trigger(processingTime="5 seconds")` | Micro-batches réguliers | ❌ Pas sur Serverless |
| `trigger(continuous="1 second")` | Mode continu expérimental | ❌ Clusters standard uniquement |

In [0]:
# arret des requêtes existantes avec le même nom
for q in spark.streams.active:
    if q.name == "orders_streaming_table":
        q.stop()

# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_queries", recurse=True)

# ecriture en memoire pour intéraction avec la requête
memory_query = stream_df.writeStream \
    .format("memory") \
    .queryName("orders_streaming_table") \
    .option("checkpointLocation", "/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_queries") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start()

In [0]:
%sql
-- maitenant on peut faire des requêtes sur une table en mémoire avec du sql
select notifications, count(*) as num_notifs
from orders_streaming_table group by notifications;

### D. Combinaison de plusieurs Streams avec UNION

In [0]:
filtered_stream1 = stream_df.filter(col("notifications") == "Y")
filtered_stream2 = stream_df.filter(col("notifications") == "N")

# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_filter", recurse=True)

combined_stream = filtered_stream1.union(filtered_stream2)

display(combined_stream, streamName="stream_filter", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_filter")

### E. Utilisation de Triggers

In [0]:
for q in spark.streams.active:
    if q.name == "triggered_query_table":
        q.stop()

# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_triggers", recurse=True)

triggered_query = stream_df \
    .withColumn("processing_ts", current_timestamp()) \
    .writeStream \
    .format("memory") \
    .queryName("triggered_query_table") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/spark_training/spark_streaming/checkpoint_triggers") \
    .trigger(availableNow=True) \
    .start()

In [0]:
%sql 
select processing_ts, count(*) as count
from triggered_query_table
group by processing_ts
order by processing_ts;